# Géoréférencement automatique par extraction du quadrillage

**PFA — Mohamed GHARBI**

Workflow en 6 étapes : détection du cadre → détection du quadrillage → saisie des 4 coins → génération automatique des GCPs → calcul de la transformation affine + écriture du `.pgw` → pipeline complet géoréférencé.

Compatible **Colab** ou **local Anaconda**.

## Setup Colab (sans effet en local)

In [ ]:
REPO_URL  = 'https://github.com/Mohamed-GHARBI/pfa.git'    # adapte
BRANCH    = 'main'
USE_DRIVE = False

import sys, subprocess, os
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
print('Environnement :', 'Google Colab' if IN_COLAB else 'Local')

if IN_COLAB:
    print('\nInstallation des dependances...')
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install',
        'opencv-python==4.10.0.84', 'scikit-image>=0.22',
        'rasterio>=1.3', 'shapely>=2.0', 'geopandas>=0.14',
        'pyogrio', 'fiona', 'ipywidgets>=8.1'])
    print('OK')
    if USE_DRIVE:
        from google.colab import drive
        drive.mount('/content/drive')
        PROJECT_ROOT = Path('/content/drive/MyDrive/pfa')
    else:
        PROJECT_ROOT = Path('/content/pfa')
        if not (PROJECT_ROOT / 'pipeline').exists():
            subprocess.check_call(['git', 'clone', '--depth', '1',
                                     '--branch', BRANCH, REPO_URL,
                                     str(PROJECT_ROOT)])
else:
    PROJECT_ROOT = Path('..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))
print(f'PROJECT_ROOT : {PROJECT_ROOT}')

In [ ]:
import cv2, numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pipeline import preprocessing as prep
from pipeline import grid_extraction as gx
from pipeline import georeferencing as geo

INPUT_PATH = str(PROJECT_ROOT / 'data' / 'raw' / 'carte_test.png')
print('Carte :', INPUT_PATH, '| existe :', os.path.exists(INPUT_PATH))

## 1. Détection du cadre cartographique

Pour le géoréférencement on travaille sur l'image PLEINE résolution (les coords pixel-monde doivent rester précises). Le `max_dimension` du notebook 01 ne s'applique pas ici.

In [ ]:
image_full = prep.load_image(INPUT_PATH)
image_full = prep.denoise(image_full)
bbox = prep.detect_map_frame(image_full)
x1, y1, x2, y2 = bbox
print(f'Cadre detecte : {bbox}'); print(f'  Crop : {x2-x1} x {y2-y1} pixels')
img_crop = image_full[y1:y2, x1:x2].copy()
img_crop_rgb = prep.to_rgb(img_crop)

## 2. Détection du quadrillage

Sur les cartes IGN/militaires tunisiennes, le quadrillage est **bleu** (Lambert Tunisia, 1 km). Pour quadrillage noir, mets `grid_color='dark'`.

In [ ]:
det = gx.detect_grid_lines(img_crop, grid_color='blue', method='projection',
                            peak_prominence=2.0, cluster_tolerance_px=30)
print('Brut         :', det)
h_filt = gx.filter_regular_lines(det.h_lines, tolerance_ratio=0.20)
v_filt = gx.filter_regular_lines(det.v_lines, tolerance_ratio=0.20)
det_filt = gx.GridDetection(h_lines=h_filt, v_lines=v_filt,
                              h_spacing_px=det.h_spacing_px,
                              v_spacing_px=det.v_spacing_px)
print('Apres filtre :', det_filt)
intersections_crop = gx.grid_intersections(det_filt)
print(f'Intersections detectees : {intersections_crop.shape[0]}')

In [ ]:
overlay = gx.draw_grid_overlay(img_crop, det_filt,
                                line_color=(0,255,0),
                                intersection_color=(255,0,255),
                                thickness=3, radius=10)
overlay_rgb = cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)
fig, ax = plt.subplots(figsize=(14, 10))
ax.imshow(overlay_rgb)
ax.set_title(f'Quadrillage : {len(det_filt.h_lines)} H, {len(det_filt.v_lines)} V, '
              f'{intersections_crop.shape[0]} intersections')
ax.axis('off'); plt.tight_layout(); plt.show()

## 3. Saisie des 4 coins

Conversion DMS -> decimal : `dd = d + m/60 + s/3600`. Exemple : 36° 48' 23" N -> 36.8064°.

In [ ]:
# REMPLACE par les vraies coords de TA carte
corners = geo.CornerCoords(
    top_left     = (10.0000, 37.0000),    # NW
    top_right    = (10.5000, 37.0000),    # NE
    bottom_right = (10.5000, 36.7500),    # SE
    bottom_left  = (10.0000, 36.7500),    # SW
)
print('Coins :', corners)

## 4. Génération automatique des GCPs

In [ ]:
intersections_orig = intersections_crop + np.array([x1, y1])
gcps = geo.gcps_from_grid_intersections(intersections_orig, bbox, corners)
print(f'GCPs generes : {len(gcps)}')
print(f'  premier : col={gcps[0].col:.0f} row={gcps[0].row:.0f}'
      f'  -> ({gcps[0].x:.6f}, {gcps[0].y:.6f})')
out_json = str(PROJECT_ROOT / 'data' / 'raw' / 'carte_test_gcps.json')
geo.save_gcps_json(gcps, out_json)
print(f'\nGCPs sauves : {out_json}')

## 5. Transformation affine + world file

In [ ]:
transform = geo.compute_transform(gcps)
print('Transformation affine :')
print(f'  a={transform.a:.10f}  b={transform.b:.10f}  c={transform.c:.10f}')
print(f'  d={transform.d:.10f}  e={transform.e:.10f}  f={transform.f:.10f}')
H_full, W_full = image_full.shape[:2]
wx, wy = geo.pixel_to_world(W_full/2, H_full/2, transform)
print(f'\nCentre image -> monde ({wx:.6f}, {wy:.6f})')
wld = geo.write_world_file(INPUT_PATH, transform)
print(f'\nWorld file : {wld}')

## 6. Pipeline complet géoréférencé

In [ ]:
from pipeline.pipeline import run_pipeline
result = run_pipeline(
    input_path=INPUT_PATH,
    output_dir=str(PROJECT_ROOT / 'data' / 'processed_geo'),
    gcps=gcps, crs='EPSG:4326',
    auto_crop=True, verbose=True,
)
print('\n=== Termine ==='); print(result.to_json())

## Vérification dans QGIS

1. Si tu es sur **Colab** : télécharge `data/processed_geo/*.geojson` et `data/raw/carte_test.png` + le `.pgw` à côté (clique droit -> Télécharger sur chaque fichier dans le panneau de gauche).
2. Ouvre QGIS -> Couche raster -> `carte_test.png`. Le `.pgw` est lu auto.
3. Couche vectorielle -> chaque `.geojson`.
4. Si tout est aligné, c'est gagné.

## Limitations

- Interpolation bilineaire (suppose projection localement lineaire). Pour 1:50000 sur 30 km, erreur < 1 m.
- Si quadrillage mal detecte : `gcps_from_corners(bbox, corners, n_samples=5)` (4 coins seuls).